# Superconductivity in VCA on a 4-site cluster

- Construct a Hubbard model based on a $2\times2$ cluster where several types of superconductivity are defined:

    - On-site, $s$-wave pairing :
    $$ \hat\Delta = \sum_\mathbf{r} ( c_{\mathbf{r}\uparrow} c_{\mathbf{r}\downarrow} + \text{H.c})$$

    - extended $s$-wave pairing : 
    $$ \hat\Delta = \sum_\mathbf{r} \big[ (c_{\mathbf{r}\uparrow} c_{\mathbf{r}+\hat{x}\downarrow} - c_{\mathbf{r}\downarrow} c_{\mathbf{r}+\hat{x}\uparrow}) + (c_{\mathbf{r}\uparrow} c_{\mathbf{r}+\hat{y}\downarrow} - c_{\mathbf{r}\downarrow} c_{\mathbf{r}+\hat{y}\uparrow}) + \text{H.c}\big] $$

    - singlet, $d_{x^2-y^2}$-wave pairing : 
    $$ \hat\Delta = \sum_\mathbf{r} \big[ (c_{\mathbf{r}\uparrow} c_{\mathbf{r}+\hat{x}\downarrow} - c_{\mathbf{r}\downarrow} c_{\mathbf{r}+\hat{x}\uparrow}) - (c_{\mathbf{r}\uparrow} c_{\mathbf{r}+\hat{y}\downarrow} - c_{\mathbf{r}\downarrow} c_{\mathbf{r}+\hat{y}\uparrow}) + \text{H.c}\big] $$

    - singlet, $d_{xy}$-wave pairing : 
    $$ \hat\Delta = \sum_\mathbf{r} \big[ (c_{\mathbf{r}\uparrow} c_{\mathbf{r}+\hat{x}+\hat{y}\downarrow} - c_{\mathbf{r}\downarrow} c_{\mathbf{r}+\hat{x}+\hat{y}\uparrow}) - (c_{\mathbf{r}\uparrow} c_{\mathbf{r}+\hat{y}-\hat{x}\downarrow} - c_{\mathbf{r}\downarrow} c_{\mathbf{r}+\hat{y}-\hat{x}\uparrow}) + \text{H.c}\big] $$

    (notation : $\hat{x}$ and $\hat{y}$ are the unit lattice vectors in the $x$ and $y$ directions, respectively.)
    
    
- Show how the Potthoff functional $\Omega$ depends on the superconducting Weiss field for each of the cases defined above.

- In the case of $d_{x^2-y^2}$-wave pairing and $U=8$, obtain the VCA solution for a hole-doped system as a function of chemical potential (or doping) for the range of doping where the solution is nontrivial. Plot the superconducting order parameter as a function of doping $x=1-n$, ($n$ begin the electron density).



In [ ]:
import pyqcm2 as pyqcm
import pyqcm2.vca as vca
import numpy as np

# declare a cluster model of 4 sites, named 'clus'
CM = pyqcm.cluster_model( 4)

# define a physical cluster based on that model, with base position (0,0,0) and site positions
clus = pyqcm.cluster(CM, ((0,0,0), (1,0,0), (0,1,0),(1,1,0))) 

# define a lattice model named '2x2' made of the cluster(s) clus and superlattice vectors (2,0,0) & (0,2,0)
model = pyqcm.lattice_model('2x2', clus, ((2,0,0), (0,2,0)))

# define a few operators in this model
model.interaction_operator('U')
model.hopping_operator('t', (1,0,0), -1)
model.hopping_operator('t', (0,1,0), -1)
model.hopping_operator('tp', (1,1,0), -1)
model.hopping_operator('tp', (-1,1,0), -1)

# On-site s-wave
model.anomalous_operator('S', [0,0,0], 1)

# Extended s-wave
model.anomalous_operator('xS', [1,0,0], 1)
model.anomalous_operator('xS', [0,1,0], 1)

# d_{x^2-y^2}-wave
model.anomalous_operator('D', [1,0,0], 1)
model.anomalous_operator('D', [0,1,0], -1)

# d_{xy}-wave, similar as previous wave but defined with "diagonal" links
model.anomalous_operator('Dxy', [1,1,0], 1)
model.anomalous_operator('Dxy', [-1,1,0], -1)

In [ ]:
# Defining the appropriate sector of Hilbert space
model.set_target_sectors('R0:S0')

# The first three parameters are used for the graph of the Potthoff functionnal
model.set_parameters("""
t=1
U=8
mu=1.2
S_1=1e-9
xS_1=1e-9
D_1=1e-9
Dxy_1=1e-9
S=0
xS=0
D=0
Dxy=0
""")

In [ ]:
# Generating a grid for the Potthoff functionnals
grid_size = 25
Delta_grid = np.linspace(1e-9, 0.3, grid_size)

In [ ]:
# The different types of superconductivity are explored by plotting the Potthoff functionnal for each variety
vca.plot_sef(model, 'S_1', Delta_grid)
model.set_parameter("S_1", 1e-9)

vca.plot_sef(model, 'xS_1', Delta_grid)
model.set_parameter('xS_1', 1e-9)

vca.plot_sef(model, 'D_1', Delta_grid)
model.set_parameter('D_1', 1e-9)

vca.plot_sef(model, 'Dxy_1', Delta_grid)
model.set_parameter('Dxy_1', 1e-9)

In [ ]:
# Loading the sef.tsv file into an array
raw_data = np.genfromtxt("./sef.tsv", delimiter="\t", names=True, dtype=None, encoding='utf8', usecols=['omega'])
data = raw_data[-4*grid_size-4:] # reading the last 4 sef simulation blocks accounting for the "omega" header

# Loading each superconducting sef into an array
omega_1 = data[0:grid_size]
omega_2 = data[grid_size:2*grid_size]
omega_3 = data[2*grid_size:3*grid_size]
omega_4 = data[3*grid_size:4*grid_size]

import matplotlib.pyplot as plt

plt.plot(Delta_grid, omega_1, label="$s$-wave")
plt.plot(Delta_grid, omega_2, label="extended $s$-wave")
plt.plot(Delta_grid, omega_3, label="$d_{x^2 - y^2}$")
plt.plot(Delta_grid, omega_4, label="$d_{xy}$")

plt.xlim((0.0, 0.3))

plt.legend()
plt.xlabel("$\Delta$")
plt.ylabel("$\Omega$")
plt.show()

### Interpretation of the Potthoff functionnals

As seen in the above plot, the only functionnal that admits a minimum is the one related to $d_{x^2 - y^2}$ superconduction. This implies that it's also the "appropriate" form of superconduction for the model we have defined.

In [ ]:
# Loop_parameters
mu_start = 1.2
mu_stop = 2.1
mu_step = 0.05

In [ ]:
import pyqcm2.vca as vca

# Setting parameters for a VCA around d_{x^2-y^2} superconduction
model.set_parameter("S_1", 1e-9)
model.set_parameter("xS_1", 1e-9)
model.set_parameter("D_1", 0.15) # Starting parameter for the VCA
model.set_parameter("Dxy_1", 1e-9)
I = pyqcm.model_instance(model)

# Defining a fucntion that runs the appropriate VCA every time func() is called in controlled_loop()
def run_vca():
    V = vca.VCA(model, varia='D_1', steps=0.01, accur=2e-3, max=100, max_iter=60, method='altNR')
    return V.I

model.controlled_loop(task=run_vca, varia='D_1', loop_param='mu', loop_range=(mu_start, mu_stop, mu_step))

In [ ]:
# Creating a grid for the plot
import numpy as np
mu_grid = np.arange(mu_start, mu_stop, mu_step)

# Loading the vca.tsv file into an array
raw_vca_data = np.genfromtxt("./vca.tsv", names=True, dtype=None, encoding='utf8')
vca_data = raw_vca_data[-len(mu_grid):] # loading the appropriate VCA simulations (last N corresponding to controlled_loop)


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

ax.plot(1-vca_data['ave_mu'], vca_data['D_1'], 'o', markersize=2, label="$\Delta$")
ax.plot(1-vca_data['ave_mu'], vca_data['ave_D'], 'o', markersize=2, label="$\langle\Delta\\rangle$")

ax.set_xlabel("$1-n$")
ax.set_ylabel("$\langle\hat\Delta\rangle$")

ax.legend()

fig.show()


### Conclusion

We can see that the most superconduction is observed for a hole density of $\approx0.095$